In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
!pip install Kmodes
from kmodes.kmodes import KModes
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances




#Read the Excel file
df = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')
df2 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
df3 = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Loyalty')

print(df.head())

In [ ]:
# Βασικός καθαρισμός A, B, C
for col in ["Category A", "Category B", "Category C"]:
    df2[col] = (
        df2[col]
        .astype(str)
        .str.strip()
        .str.title()
    )
    df2[col] = df2[col].replace(["Nan", "None", "Na", ""], np.nan)

# 2️⃣ Δημιουργούμε νέα στήλη CustomCategory (ξεκινάει ίδια με Category B)
df2["CustomCategory"] = df2["Category B"].copy()

# =========================================================
# 3️⃣ ΟΛΕΣ ΟΙ ΠΑΛΙΕΣ ΑΛΛΑΓΕΣ ΣΟΥ ΠΑΝΩ ΣΤΗΝ CustomCategory
# =========================================================

# 3.1 Συσκευασμενο → "Category C + ' σε συσκευασία'"
mask_sysk = df2["CustomCategory"] == "Συσκευασμενο"
df2.loc[mask_sysk, "CustomCategory"] = (
    df2.loc[mask_sysk, "Category C"] + " σε συσκευασία"
)

# 3.2 Merge γαλακτοκομικών σε ενιαία κατηγορία (θα το σπάσουμε μετά)
to_merge_dairy = [
    "Γιαουρτια σε συσκευασία",
    "Τυροκομικα σε συσκευασία",
    "Γαλατα σε συσκευασία",
    "Βουτυρα σε συσκευασία",
    "Κρεμα Γαλακτος σε συσκευασία",
]
df2["CustomCategory"] = df2["CustomCategory"].replace(
    to_merge_dairy, "Γαλακτοκομικά σε συσκευασία"
)

# 3.3 Ρουχων + Ενδυση → Ρούχα & Ενδυση
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση"
)

# 3.4 Μπυρες + Κρασια + Οινοπνευματωδη → Οινοπνευματωδη
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη"
)

# 3.5 Σωματος / Ξυριστικα / Χεριων / Προσωπου → Προϊόντα Προσωπικής Φροντίδας
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"],
    "Προϊόντα Προσωπικής Φροντίδας",
)

# 3.6 Βαμβακια / Πανες Ακρατειας → Προιοντα Χαρτου
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου"
)

# 3.7 Μωρομαντηλα / Πανες Παιδικες → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Μωρομαντηλα", "Πανες Παιδικες"], "Παιδικα"
)

# 3.8 Βρεφικη Τροφη → Παιδικα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Βρεφικη Τροφη", "Παιδικα"
)

# 3.9 Χυμοι / Ροφηματα → Χυμοί & Ροφήματα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία"],
    "Χυμοί & Ροφήματα",
)
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Αναψυκτικα", "Χυμοί & Ροφήματα"
)

# 3.10 Κρεας σε συσκευασία → Κατεψυγμενα
df2["CustomCategory"] = df2["CustomCategory"].replace(
    "Κρεας σε συσκευασία", "Κατεψυγμενα"
)

# 3.11 Σαλτσες / Dressings → Σάλτσες & Dressings
df2["CustomCategory"] = df2["CustomCategory"].replace(
    ["Σαλτσες", "Dressings"], "Σάλτσες & Dressings"
)

# =========================================================
# 4️⃣ ΟΛΕΣ ΟΙ ΝΕΕΣ "ΕΞΥΠΝΕΣ" ΑΛΛΑΓΕΣ ΑΠΟ ΤΗΝ ΑΝΑΛΥΣΗ
# =========================================================

# 4.1 Split Ρούχα & Ενδυση → Προϊόντα Πλυντηρίου Ρούχων vs Ρούχα
laundry_items = [
    "Υγρα Πλυντηριου",
    "Μαλακτικα Πλυντηριου",
    "Ενισχυτικα-Χρωμοπαγιδες",
    "Σκονη Πλυντηριου",
    "Ταμπλετες Πλυντηριου",
    "Αποσκληρυντικα Πλυντηριου",
    "Πλυσιμο Στο Χερι",
    "Σιδερωματος",
]

mask_laundry = (
    (df2["CustomCategory"] == "Ρούχα & Ενδυση")
& (df2["Category C"].isin(laundry_items))
)
df2.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"

# 4.2 Διάλυση "Χυμα" σε λογικές κατηγορίες
xuma_map = {
    "Τυροκομικα": "Γαλακτοκομικά σε συσκευασία",
    "Αλλαντικα": "Αλλαντικα σε συσκευασία",
    "Μαναβικη": "Μαναβικη σε συσκευασία",
    "Ξηροι Καρποι": "Αλμυρα Σνακ",
    "Χαλβας": "Χαλβαδες Ταχινι",
    "Αλιπαστα": "Κονσερβες",
    "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",
}
mask_xuma = df2["Category B"] == "Χυμα"
df2.loc[mask_xuma, "CustomCategory"] = (
    df2.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα")
)

# 4.3 Αλευρι από Αρτοσκευασματα → Βασικά Υλικά Μαγειρικής
mask_alevri = (df2["Category B"] == "Αρτοσκευασματα") & (df2["Category C"] == "Αλευρι")
df2.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"

# 4.4 Σπάσιμο Γαλακτοκομικών σε επιμέρους κατηγορίες
dairy_split_map = {
    "Γιαουρτια": "Γιαουρτια",
    "Τυροκομικα": "Τυροκομικα",
    "Γαλατα": "Γαλατα",
    "Βουτυρα": "Βουτυρα",
    "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",
}
mask_dairy = df2["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
df2.loc[mask_dairy, "CustomCategory"] = (
    df2.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία")
)

# 4.5 Split Γλυκα Σνακ
mask_glyka = df2["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = df2["Category C"]

df2.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
df2.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
df2.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
df2.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
df2.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
df2.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"

# 4.6 Split Πρωινο
mask_proino = df2["CustomCategory"] == "Πρωινο"
c_pro = df2["Category C"]

# Ροφήματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]),
         "CustomCategory"] = "Ροφηματα Πρωινου"

# Δημητριακά
df2.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"

# Αλείμματα πρωινού
df2.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]),
         "CustomCategory"] = "Αλειμματα Πρωινου"

# Εβαπορε → Γαλατα
df2.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"

# 4.7 Split Χυμοί & Ροφήματα
mask_drinks = df2["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = df2["Category C"]

# Αναψυκτικά
df2.loc[mask_drinks & c_dr.isin(
    ["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]
), "CustomCategory"] = "Αναψυκτικα"

# Χυμοί & Νέκταρ
df2.loc[mask_drinks & c_dr.isin(
    ["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]
), "CustomCategory"] = "Χυμοι & Νεκταρ"

# Έτοιμα ροφήματα (Ice Tea, Ice Coffee κ.λπ.)
df2.loc[mask_drinks & c_dr.isin(
    ["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]
), "CustomCategory"] = "Rtd Ροφηματα"

# Ενεργειακά
df2.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"

# 4.8 Split Κατεψυγμενα
mask_frozen = df2["CustomCategory"] == "Κατεψυγμενα"
c_fr = df2["Category C"]

df2.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
df2.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
df2.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
df2.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
df2.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"

# 4.9 Split Αλμυρα Σνακ → Ξηροι Καρποι ξεχωριστά
mask_salty = df2["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = df2["Category C"]

df2.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"

# =========================================================
# 5️⃣ Κανόνας: μικρές κατηγορίες (<10 barcodes) → "Διαφορα"
# =========================================================
counts = df2["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
df2["CustomCategory"] = df2["CustomCategory"].replace(small_cats, "Διαφορα")

# Ζυμες Ψυγειου σε συσκευασία → Αρτοσκευασματα
df2.loc[df2["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"

# =========================================================
# 6️⃣ Γρήγορος έλεγχος
# =========================================================
print("Μοναδικές Category B       :", df2["Category B"].nunique())
print("Μοναδικές CustomCategory   :", df2["CustomCategory"].nunique())
print("\nTop 40 CustomCategory:")
print(df2["CustomCategory"].value_counts().head(40))

# Νέα ενότητα

In [ ]:
#exclude quantities < 1
print(df.shape)
df = df[df['Quantity'] >= 1]
print(df.shape)

In [ ]:
display(df.describe())

In [ ]:
#exclude non positive values
df = df[df['Value_'] > 0]
print(df.shape)

In [ ]:
#exclude non integer quantities
df = df[df['Quantity'] % 1 == 0]
print(df.shape)

In [ ]:
#create new column Price
df['Price'] = df['Value_'] / df['Quantity']
print(df.shape)

In [ ]:
#Remove baskets with no LoyaltyCard attached
df = df[df['LoyaltyCard_ID'].notna()]
print(df.shape)


In [ ]:
df = df[df['Value_'].notna()]
df = df[df['Barcode'].notna()]
df = df[df['Date_'].notna()]
df = df[df['Basket_ID'].notna()]
df = df[df['Quantity'].notna()]
print(df.shape)
df.head()

In [ ]:
#Exclude barcodes that are not contained in the list of real barcodes
df = df[df['Barcode'].isin(df2['Barcode'])]
print(df.shape)

In [ ]:
#Exclude transactions that did not contain cardid's where cardholder was known or his Status was na
# try to fix the na into NaN?
valid_cards = df3.loc[df3['Status'].str.contains('na', na=False), 'Cardholder']

df = df[~df['LoyaltyCard_ID'].isin(valid_cards)]
print(df.shape)

In [ ]:
#Convert date from string to datetime
df['Date_'] = pd.to_datetime(df['Date_'], errors='coerce', dayfirst=True)
df.head()

In [ ]:
df.replace(['na'], pd.NA, inplace=True)
df.describe()


In [ ]:
plt.hist(df['Quantity'], bins=20, edgecolor='black')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.title('Distribution of Quantity')
plt.show()

In [ ]:
# Do we delete?
#Q1 = df['Quantity'].quantile(0.25)
#Q3 = df['Quantity'].quantile(0.75)
#IQR = Q3 - Q1

#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
#df = df[(df['Quantity'] >= lower_bound) & (df['Quantity'] <= upper_bound)]
#print(df.shape)

In [ ]:
plt.hist(df['Quantity'], bins=20, edgecolor='black')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.title('Distribution of Quantity')
plt.show()

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
Q1 = df['Value_'].quantile(0.25)
Q3 = df['Value_'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter the DataFrame
df = df[(df['Value_'] >= lower_bound) & (df['Value_'] <= upper_bound)]
print(df.shape)

In [ ]:
plt.hist(df['Value_'], bins=20, edgecolor='black')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Distribution of Value')
plt.show()

In [ ]:
df.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
df = pd.merge(df, df3, left_on='LoyaltyCard_ID', right_on='Cardholder', how='inner')
df.head()

In [ ]:
#df2['Custom_Category'] = (
    #df2['Category C']
    #.combine_first(df2['Category B'])
    #.combine_first(df2['Category A'])
#)

In [ ]:
#df2.head()
#df2['Custom_Category'].nunique()
#df2['Category C'].nunique()
#df_counts = df2['Category C'].value_counts().reset_index()
#df_counts.columns = ['Category_C', 'Count']
#print(df_counts)

In [ ]:
#df2 = pd.merge(df2, df_counts, left_on='Custom_Category', right_on='Category_C', how='inner')
#df2.head()

In [ ]:
#df2['Custom_Category'] = np.where(df2['Count'] > 60, df2['Category C'], df2['Category B'])
#df2.head()
#df2['Custom_Category'].nunique()

In [ ]:
df = pd.merge(df, df2, left_on='Barcode', right_on='Barcode', how='inner')
df.head()

In [ ]:
df.drop(columns=['Category A', 'Category B', 'Category C','Category C'], inplace=True)
df.head()

In [ ]:
#df.to_excel("/content/drive/MyDrive/output.xlsx", index=False)

In [ ]:
#Create One-hot encoding for the data to be ready for clustering
basket_matrix = pd.crosstab(df['Basket_ID'], df['CustomCategory'])
basket_matrix = (basket_matrix > 0).astype(int)
print(basket_matrix)

In [ ]:
basket_matrix['Total_categories'] = basket_matrix.sum(axis=1)
basket_matrix.head()

In [ ]:
basket_matrix['Total_categories'].describe()

In [ ]:
# Filter out the 1 item baskets
basket_matrix = basket_matrix[(basket_matrix['Total_categories'] > 1)]
print(basket_matrix.shape)

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

kmeans = KMeans(random_state=0)
visualizer = KElbowVisualizer(kmeans, k=(1,11))

visualizer.fit(basket_matrix.drop(columns=['Total_categories']))
_ = visualizer.show()

In [ ]:
data = basket_matrix.drop(columns=['Total_categories'])

# Calculate costs for different k values
costs = []
K_range = range(2, 7)  # 1 to 10 clusters

for k in K_range:
    # For k=1, we can use a simple approach or skip since one cluster is trivial
    if k == 1:
        # For single cluster, cost is total dissimilarity from mode
        kmodes = KModes(n_clusters=1, init='Huang', n_init=5, random_state=0)
        kmodes.fit(data)
        costs.append(kmodes.cost_)
    else:
        kmodes = KModes(n_clusters=k, init='Huang', n_init=5, random_state=0)
        kmodes.fit(data)
        costs.append(kmodes.cost_)

# Plot the elbow curve
plt.figure(figsize=(8, 6))
plt.plot(K_range, costs, 'bx-', linewidth=2, markersize=8)
plt.xlabel('Number of clusters (k)', fontsize=12)
plt.ylabel('Cost (within-cluster dissimilarity)', fontsize=12)
plt.title('Elbow Method for k-modes', fontsize=14)
plt.xticks(K_range)
plt.grid(True, alpha=0.3)

# Mark the elbow point (you can do this visually or calculate it)
plt.show()

In [ ]:
#from yellowbrick.cluster import SilhouetteVisualizer

#plt.figure(figsize=(2 * 5,  10 * 4))

#scores = {}
#for n_clusters in range(2, 20):
    #plt.subplot(10, 2, n_clusters - 1)
    #kmeans = KMeans(n_clusters, random_state=42)
    #visualizer = SilhouetteVisualizer(kmeans, colors='yellowbrick')
    #visualizer.fit(basket_matrix.drop(columns=['Total_categories']))
    #scores[n_clusters] = visualizer.silhouette_score_
    #plt.title(f'clusters: {n_clusters} score: {visualizer.silhouette_score_}')

In [ ]:
#sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

In [ ]:
kmeans = KModes(n_clusters=4, n_init=10, random_state=42)
kmeans.fit(basket_matrix.drop(columns=['Total_categories']))

In [ ]:
#dist = pairwise_distances(basket_matrix.drop(columns=['Total_categories']).astype(bool).values, metric='jaccard')
#db = DBSCAN(eps=0.3, min_samples=5, metric='precomputed')
#basket_matrix['cluster'] = db.fit_predict(dist)

In [ ]:
basket_matrix['cluster'] = kmeans.labels_
print(basket_matrix['cluster'])

In [ ]:
basket_matrix['cluster'].value_counts()

In [ ]:
cluster_profiles = basket_matrix.groupby('cluster').mean().drop(columns=['cluster','Total_categories'], errors='ignore')

In [ ]:
cluster_profiles_no_dairy = cluster_profiles.drop(columns=['Τυροκομικα'], errors='ignore')

filtered_profiles_dict = {
    cluster: row[row >= 0.10].sort_values(ascending=False)
    for cluster, row in cluster_profiles_no_dairy.iterrows()
}

for cluster, series in filtered_profiles_dict.items():
    if not series.empty: # Only plot if there are categories meeting the threshold
        plt.figure(figsize=(8,4))
        series.plot(kind='bar')
        plt.title(f"Cluster {cluster} (No Dairy): Product Categories \u226510% Presence")
        plt.ylabel("Percentage of Baskets")
        plt.ylim(0, 1)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No categories met the >=10% presence threshold for Cluster {cluster} (No Dairy).")

In [ ]:
!pip install umap-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import time

# ====================================================================
# 0. ΟΡΙΣΜΟΣ ΑΡΧΕΙΩΝ & ΠΡΟΕΤΟΙΜΑΣΙΑ
# ====================================================================

# Χρησιμοποιούμε τα ονόματα αρχείων CSV ως strings
FILE_HIER = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx', sheet_name = 'Hierachy Categories & Barcodes')
FILE_POS = pd.read_excel('/content/POS_DATA_BAPR_2024-2025_updated (1).xlsx')


In [ ]:
# 1. Φόρτωμα & Καθαρισμός Ιεραρχίας
hier = FILE_HIER.copy()  # Χρησιμοποίησε το DataFrame που είναι ήδη φορτωμένο
for col in ["Category A", "Category B", "Category C"]:
    hier[col] = hier[col].astype(str).str.strip().str.title()
    hier[col] = hier[col].replace(["Nan", "None", "Na", ""], np.nan)
hier["CustomCategory"] = hier["Category B"].copy()

# ... (ΟΛΟΙ ΟΙ ΚΑΝΟΝΕΣ CUSTOM CATEGORY) ...
mask_sysk = hier["CustomCategory"] == "Συσκευασμενο"
hier.loc[mask_sysk, "CustomCategory"] = hier.loc[mask_sysk, "Category C"] + " σε συσκευασία"
to_merge_dairy = ["Γιαουρτια σε συσκευασία", "Τυροκομικα σε συσκευασία", "Γαλατα σε συσκευασία", "Βουτυρα σε συσκευασία", "Κρεμα Γαλακτος σε συσκευασία", ]
hier["CustomCategory"] = hier["CustomCategory"].replace(to_merge_dairy, "Γαλακτοκομικά σε συσκευασία")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Ρουχων", "Ενδυση"], "Ρούχα & Ενδυση")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μπυρες", "Κρασια", "Οινοπνευματωδη"], "Οινοπνευματωδη")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σωματος", "Ξυριστικα", "Χεριων", "Προσωπου"], "Προϊόντα Προσωπικής Φροντίδας")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Βαμβακια", "Πανες Ακρατειας"], "Προιοντα Χαρτου")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Μωρομαντηλα", "Πανες Παιδικες", "Βρεφικη Τροφη"], "Παιδικα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Χυμοι - Τσαι Ψυγειου σε συσκευασία", "Χυμοι", "Ροφηματα σε συσκευασία", "Αναψυκτικα"], "Χυμοί & Ροφήματα")
hier["CustomCategory"] = hier["CustomCategory"].replace("Κρεας σε συσκευασία", "Κατεψυγμενα")
hier["CustomCategory"] = hier["CustomCategory"].replace(["Σαλτσες", "Dressings"], "Σάλτσες & Dressings")
laundry_items = ["Υγρα Πλυντηριου", "Μαλακτικα Πλυντηριου", "Ενισχυτικα-Χρωμοπαγιδες", "Σκονη Πλυντηριου", "Ταμπλετες Πλυντηριου", "Αποσκληρυντικα Πλυντηριου", "Πλυσιμο Στο Χερι", "Σιδερωματος", ]
mask_laundry = ((hier["CustomCategory"] == "Ρούχα & Ενδυση") & (hier["Category C"].isin(laundry_items)))
hier.loc[mask_laundry, "CustomCategory"] = "Προϊόντα Πλυντηρίου Ρούχων"
xuma_map = {"Τυροκομικα": "Γαλακτοκομικά σε συσκευασία", "Αλλαντικα": "Αλλαντικα σε συσκευασία", "Μαναβικη": "Μαναβικη σε συσκευασία", "Ξηροι Καρποι": "Αλμυρα Σνακ", "Χαλβας": "Χαλβαδες Ταχινι", "Αλιπαστα": "Κονσερβες", "Βουτυρα": "Γαλακτοκομικά σε συσκευασία",}
mask_xuma = hier["Category B"] == "Χυμα"
hier.loc[mask_xuma, "CustomCategory"] = (hier.loc[mask_xuma, "Category C"].map(xuma_map).fillna("Χυμα"))
mask_alevri = (hier["Category B"] == "Αρτοσκευασματα") & (hier["Category C"] == "Αλευρι")
hier.loc[mask_alevri, "CustomCategory"] = "Βασικά Υλικά Μαγειρικής"
dairy_split_map = {"Γιαουρτια": "Γιαουρτια", "Τυροκομικα": "Τυροκομικα", "Γαλατα": "Γαλατα", "Βουτυρα": "Βουτυρα", "Κρεμα Γαλακτος": "Κρεμα Γαλακτος",}
mask_dairy = hier["CustomCategory"] == "Γαλακτοκομικά σε συσκευασία"
hier.loc[mask_dairy, "CustomCategory"] = (hier.loc[mask_dairy, "Category C"].map(dairy_split_map).fillna("Γαλακτοκομικά σε συσκευασία"))
mask_glyka = hier["CustomCategory"] == "Γλυκα Σνακ"
c_glyka = hier["Category C"]
hier.loc[mask_glyka & c_glyka.isin(["Μπισκοτα", "Wafer"]), "CustomCategory"] = "Μπισκοτα & Wafers"
hier.loc[mask_glyka & c_glyka.isin(["Σοκολατες", "Ζαχαρωδη"]), "CustomCategory"] = "Σοκολατες & Ζαχαρωδη"
hier.loc[mask_glyka & (c_glyka == "Κρουασαν"), "CustomCategory"] = "Κρουασαν & Bake Snacks"
hier.loc[mask_glyka & c_glyka.isin(["Παραδοσιακα Γλυκισματα", "Κεικ"]), "CustomCategory"] = "Ειδη Ζαχαροπλαστικης"
hier.loc[mask_glyka & (c_glyka == "Τσουρεκι"), "CustomCategory"] = "Αρτοσκευασματα"
hier.loc[mask_glyka & (c_glyka == "Εποχιακα"), "CustomCategory"] = "Εποχιακα Ειδη"
mask_proino = hier["CustomCategory"] == "Πρωινο"
c_pro = hier["Category C"]
hier.loc[mask_proino & c_pro.isin(["Καφες", "Τσαι", "Σοκολατουχα Ροφηματα", "Αρωματικα Ροφηματα"]), "CustomCategory"] = "Ροφηματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Δημητριακα"), "CustomCategory"] = "Δημητριακα Πρωινου"
hier.loc[mask_proino & c_pro.isin(["Μαρμελαδες", "Μελι", "Πραλινες Spreads", "Peanut Butter Spreads"]), "CustomCategory"] = "Αλειμματα Πρωινου"
hier.loc[mask_proino & (c_pro == "Εβαπορε"), "CustomCategory"] = "Γαλατα"
mask_drinks = hier["CustomCategory"] == "Χυμοί & Ροφήματα"
c_dr = hier["Category C"]
hier.loc[mask_drinks & c_dr.isin(["Cola", "Πορτοκαλαδα", "Λεμοναδα", "Γκαζοζα", "Soda Tonik Mixers", "Various Flavours"]), "CustomCategory"] = "Αναψυκτικα"
hier.loc[mask_drinks & c_dr.isin(["Φυσικοι", "Νεκταρ", "Φρουτοποτα", "Συμπυκνωμενοι"]), "CustomCategory"] = "Χυμοι & Νεκταρ"
hier.loc[mask_drinks & c_dr.isin(["Ice Tea Ice Coffee", "Ροφηματα", "Χυμοι - Τσαι Ψυγειου"]), "CustomCategory"] = "Rtd Ροφηματα"
hier.loc[mask_drinks & (c_dr == "Ενεργειακα Ισοτονικα"), "CustomCategory"] = "Ενεργειακα & Ισοτονικα"
mask_frozen = hier["CustomCategory"] == "Κατεψυγμενα"
c_fr = hier["Category C"]
hier.loc[mask_frozen & (c_fr == "Παγωτα"), "CustomCategory"] = "Κατεψυγμενα Παγωτα"
hier.loc[mask_frozen & (c_fr == "Ζυμες"), "CustomCategory"] = "Κατεψυγμενες Ζυμες"
hier.loc[mask_frozen & (c_fr == "Ψαρικα"), "CustomCategory"] = "Κατεψυγμενα Ψαρικα"
hier.loc[mask_frozen & (c_fr == "Λαχανικα"), "CustomCategory"] = "Κατεψυγμενα Λαχανικα"
hier.loc[mask_frozen & c_fr.isin(["Κρεας", "Ετοιμα Φαγητα"]), "CustomCategory"] = "Κατεψυγμενα Κρεας & Γευματα"
mask_salty = hier["CustomCategory"] == "Αλμυρα Σνακ"
c_salty = hier["Category C"]
hier.loc[mask_salty & (c_salty == "Ξηροι Καρποι"), "CustomCategory"] = "Ξηροι Καρποι"
counts = hier["CustomCategory"].value_counts()
small_cats = counts[counts < 10].index.tolist()
hier["CustomCategory"] = hier["CustomCategory"].replace(small_cats, "Διαφορα")
hier.loc[hier["CustomCategory"] == "Ζυμες Ψυγειου σε συσκευασία", "CustomCategory"] = "Αρτοσκευασματα"




In [ ]:
# 2. Φόρτωμα POS & Καθαρισμός
df_pos = FILE_POS.copy()  # Χρησιμοποίησε το DataFrame αντί CSV
df_pos = df_pos.rename(columns={"Value_": "Value", "Date_": "Date"}, errors="ignore")
df_pos['Date'] = pd.to_datetime(df_pos['Date'], errors='coerce', dayfirst=True)
df_pos = df_pos.merge(hier[["Barcode", "CustomCategory"]], on="Barcode", how="left")
df_pos = df_pos[(df_pos["Quantity"] > 0) & (df_pos["Value"] > 0)].copy()
df_pos = df_pos[df_pos["Quantity"] % 1 == 0]

In [ ]:

# 3. Basket Segmentation (Q1 Features - KMeans on Scaled Data)
basket_feats = df_pos.groupby("Basket_ID").agg(
    Total_Value=("Value", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Unique_Items=("Barcode", "nunique")
)
cat_value = df_pos.groupby(["Basket_ID", "CustomCategory"])["Value"].sum().unstack(fill_value=0)
cat_share = cat_value.div(cat_value.sum(axis=1).replace(0, 1), axis=0)
cat_share.columns = [f"Share_{c}" for c in cat_share.columns]
final_basket_df = basket_feats.join(cat_share, how="left").fillna(0)
share_cols = [c for c in final_basket_df.columns if c.startswith("Share_")]

# PCA & K-Means (K=3 for Baskets)
X_base = StandardScaler().fit_transform(final_basket_df[["Total_Value","Total_Quantity","Unique_Items"]])
X_shares_scaled = StandardScaler().fit_transform(final_basket_df[share_cols])
pca = PCA(n_components=0.80, random_state=0).fit(X_shares_scaled)
X = np.hstack([X_base, pca.transform(X_shares_scaled)])
kmeans_basket = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
final_basket_df["Cluster"] = kmeans_basket.labels_



In [ ]:

# 4. Customer Segmentation Features
df_loyal = df_pos.dropna(subset=["LoyaltyCard_ID"])
df_loyal = df_loyal.merge(final_basket_df[["Cluster"]], on="Basket_ID", how="left")

# R: Recency
current_date = df_loyal['Date'].max() + pd.Timedelta(days=1)
recency_df = df_loyal.groupby('LoyaltyCard_ID')['Date'].max().reset_index()
recency_df['Recency'] = (current_date - recency_df['Date']).dt.days

# F, M, Avg Basket Value, Avg Unique Items, Basket Shares
customer_baskets = df_loyal.groupby(['LoyaltyCard_ID', 'Basket_ID']).agg(
    Total_Basket_Value=('Value', 'sum'),
    Cluster=('Cluster', 'first')
).reset_index()

customer_feats = customer_baskets.groupby('LoyaltyCard_ID').agg(
    Frequency=('Basket_ID', 'nunique'),
    Total_Monetary=('Total_Basket_Value', 'sum'),
    Avg_Basket_Value=('Total_Basket_Value', 'mean')
)
customer_feats = customer_feats.join(df_loyal.groupby('LoyaltyCard_ID')['Barcode'].nunique().rename('Avg_Unique_Items_Per_Basket_Total'), how='inner')
customer_feats = customer_feats.join(recency_df.set_index('LoyaltyCard_ID')[['Recency']], how='inner')

cluster_counts = customer_baskets.groupby(['LoyaltyCard_ID', 'Cluster'])['Basket_ID'].nunique().unstack(fill_value=0)
total_baskets = customer_feats['Frequency']
cluster_shares = cluster_counts.div(total_baskets, axis=0)
cluster_shares.columns = [f"Share_Cluster_{c}" for c in sorted(cluster_counts.columns.tolist())]

final_customer_df = customer_feats.join(cluster_shares, how='inner').fillna(0)
final_customer_df = final_customer_df.drop(columns=['Total_Monetary'], errors='ignore') 



In [ ]:
# 5. K-Selection (K=4) - ΜΕ ΚΑΝΟΝΙΚΟΠΟΙΗΣΗ
customer_cols_for_scaling = [
    'Recency', 'Frequency', 'Avg_Basket_Value', 'Avg_Unique_Items_Per_Basket_Total'
] + cluster_shares.columns.tolist()

X_cust = final_customer_df[customer_cols_for_scaling].values

# Κανονικοποίηση σε [0, 1]
from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
X_cust_normalized = scaler_minmax.fit_transform(X_cust)

# ====== PRINT: Έλεγχος κανονικοποίησης ======
print("\n" + "="*80)
print("--- NORMALIZATION CHECK (Min-Max [0, 1]) ---")
print("="*80)
print(f"Shape of normalized data: {X_cust_normalized.shape}")
print(f"\nMin values per feature: {X_cust_normalized.min(axis=0)}")
print(f"Max values per feature: {X_cust_normalized.max(axis=0)}")
print(f"\nFirst 5 rows of normalized data:")
print(pd.DataFrame(X_cust_normalized[:5], columns=customer_cols_for_scaling).round(4))
print(f"\nData statistics after normalization:")
print(pd.DataFrame(X_cust_normalized, columns=customer_cols_for_scaling).describe().round(4))

# Επιπλέον StandardScaler για K-Means
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust_normalized)

print("\n" + "="*80)
print("--- STANDARDIZATION CHECK (StandardScaler) ---")
print("="*80)
print(f"Shape of scaled data: {X_cust_scaled.shape}")
print(f"Mean values per feature (should be ~0): {X_cust_scaled.mean(axis=0).round(6)}")
print(f"Std dev per feature (should be ~1): {X_cust_scaled.std(axis=0).round(6)}")



inertias_cust = []
sil_scores_cust = []
K_range_cust = range(3, 12)

print("\n" + "="*80)
print("--- K-SELECTION ANALYSIS ---")
print("="*80)
print(f"{'k':<5} {'Inertia':<15} {'Silhouette Score':<20}")
print("-" * 40)

for k in K_range_cust:
    mbk_cust = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_cust_scaled)
    inertias_cust.append(mbk_cust.inertia_)
    sample_size = min(5000, X_cust_scaled.shape[0])
    sil = silhouette_score(X_cust_scaled, mbk_cust.labels_, sample_size=sample_size, random_state=42)
    sil_scores_cust.append(sil)
    
    print(f"{k:<5} {inertias_cust[-1]:<15.4f} {sil_scores_cust[-1]:<20.6f}")

if len(sil_scores_cust) > 0:
    best_idx = int(np.argmax(sil_scores_cust))
    k_final_cust = list(K_range_cust)[best_idx]
    print(f"\nBest k by Silhouette: {k_final_cust} (score: {sil_scores_cust[best_idx]:.4f})")
else:
    k_final_cust = 4
    print(f"\nNo silhouette scores available — using fallback k = {k_final_cust}")

print("-" * 40)
print(f"✓ Selected k = {k_final_cust}")
print("="*80)

In [ ]:
# 5.1 Αφαίρεση Outliers (B2B/Internal Accounts)
print("\n" + "="*80)
print("--- OUTLIER DETECTION & REMOVAL ---")
print("="*80)

# Outliers: Frequency > 1000 ή Recency < 2
outlier_mask = (final_customer_df['Frequency'] > 1000) | (final_customer_df['Recency'] < 2)
n_outliers = outlier_mask.sum()
print(f"Detected {n_outliers} outliers (B2B/Internal accounts)")
if n_outliers > 0:
    print("\nOutlier details:")
    print(final_customer_df[outlier_mask][['Recency', 'Frequency', 'Avg_Basket_Value', 'Avg_Unique_Items_Per_Basket_Total']])

# Remove outliers
final_customer_df = final_customer_df[~outlier_mask].copy()
X_cust = final_customer_df[customer_cols_for_scaling].values

# Ξαναδημιουργούμε κανονικοποίηση & standardization
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
X_cust_normalized = scaler_minmax.fit_transform(X_cust)

scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust_normalized)

print(f"\n✓ Outliers removed. New customer count: {len(final_customer_df)}")
print("="*80)

In [ ]:
# ...existing code...
# --- Re-run K-selection AFTER outlier removal ---
inertias_cust = []
sil_scores_cust = []
K_range_cust = range(2, 12)

for k in K_range_cust:
    mbk_cust = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3).fit(X_cust_scaled)
    inertias_cust.append(mbk_cust.inertia_)
    sample_size = min(5000, X_cust_scaled.shape[0])
    sil = silhouette_score(X_cust_scaled, mbk_cust.labels_, sample_size=sample_size, random_state=42)
    sil_scores_cust.append(sil)
    print(f"{k:<5} {inertias_cust[-1]:<15.4f} {sil_scores_cust[-1]:<20.6f}")

if len(sil_scores_cust) > 0:
    best_idx = int(np.argmax(sil_scores_cust))
    k_final_cust = list(K_range_cust)[best_idx]
else:
    k_final_cust = 4

print(f"✓ Selected k = {k_final_cust}")

# --- Safe dynamic cluster names ---
base_names = ["Mainstream", "Stock-Up", "At-Risk", "Premium", "VIP"]
if k_final_cust <= len(base_names):
    cluster_names = {i: base_names[i] for i in range(k_final_cust)}
else:
    cluster_names = {i: (base_names[i] if i < len(base_names) else f"Cluster_{i}") for i in range(k_final_cust)}
# ...existing code...

In [ ]:
# 5.5 UMAP Visualization (ΠΡΟΣΘΗΚΗ)
import umap

print("\n" + "="*80)
print("--- UMAP DIMENSIONALITY REDUCTION ---")
print("="*80)

umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_cust_umap = umap_reducer.fit_transform(X_cust_scaled)

print(f"✓ UMAP transformation complete: {X_cust_umap.shape}")
print("="*80)

In [ ]:
# 5.6 K-Selection Plots (Elbow + Silhouette)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Elbow Curve
axes[0].plot(list(K_range_cust), inertias_cust, '-o', linewidth=2.5, markersize=10, color='steelblue')
axes[0].axvline(x=k_final_cust, color='red', linestyle='--', linewidth=2.5, label=f'Selected k={k_final_cust}')
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method - Customer Segmentation', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)
axes[0].set_xticks(list(K_range_cust))

# Subplot 2: Silhouette Score
axes[1].plot(list(K_range_cust), sil_scores_cust, '-o', linewidth=2.5, markersize=10, color='coral')
axes[1].axvline(x=k_final_cust, color='red', linestyle='--', linewidth=2.5, label=f'Selected k={k_final_cust}')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score - Customer Segmentation', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)
axes[1].set_xticks(list(K_range_cust))

plt.tight_layout()
plt.savefig("customer_k_selection_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ K-Selection plots saved: customer_k_selection_analysis.png")

In [ ]:
# Cell #VSC-5a7565f1 - FIX

# 6. UMAP Visualization με Clusters
kmeans_cust_temp = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
cluster_labels_umap = kmeans_cust_temp.labels_

# ✅ Δυναμικά ονόματα
cluster_names_list = ["Mainstream", "Stock-Up", "At-Risk", "Premium", "VIP"]
cluster_names = {i: (cluster_names_list[i] if i < len(cluster_names_list) else f"Cluster_{i}")
                 for i in range(k_final_cust)}

# UMAP scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_cust_umap[:, 0], X_cust_umap[:, 1], 
                     c=cluster_labels_umap, cmap='viridis', 
                     s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
cbar = plt.colorbar(scatter, label=f'Customer Cluster (k={k_final_cust})')
cbar.set_ticks(range(k_final_cust))
cbar.set_ticklabels([cluster_names.get(i, f"C{i}") for i in range(k_final_cust)])

plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.title(f'Customer Segmentation - UMAP Visualization (k={k_final_cust})', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("customer_umap_clusters.png", dpi=150)
plt.show()

print(f"✓ UMAP Visualization: {len(np.unique(cluster_labels_umap))} clusters detected")
print(f"  Cluster distribution:\n{pd.Series(cluster_labels_umap).value_counts().sort_index()}")

In [ ]:
# Cell #VSC-fe55aed8 - FIX

kmeans_cust = KMeans(n_clusters=k_final_cust, random_state=42, n_init=10).fit(X_cust_scaled)
final_customer_df["Customer_Cluster"] = kmeans_cust.labels_

# ✅ Δυναμικό aggregation
agg_dict = {
    "Count": ("Customer_Cluster", "size"),
    "Recency_Days_Avg": ("Recency", "mean"),
    "Frequency_Baskets_Avg": ("Frequency", "mean"),
    "Avg_Basket_Value_Avg": ("Avg_Basket_Value", "mean"),
    "Avg_Unique_Items_Total_Avg": ("Avg_Unique_Items_Per_Basket_Total", "mean"),
}

# Προσθήκη Share_Cluster_* στατικά
for col in final_customer_df.columns:
    if col.startswith("Share_Cluster_"):
        agg_dict[f"{col}_Avg"] = (col, "mean")

customer_summary = final_customer_df.groupby("Customer_Cluster").agg(**agg_dict).round(2)

print("\n" + "="*80)
print(f"--- Τελική Περίληψη Customer Segments (k={k_final_cust}) ---")
print("="*80)
print(customer_summary)

In [ ]:
# 7.1 Customer Cluster Profiles Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Plot 1: Cluster Size
cluster_sizes = final_customer_df["Customer_Cluster"].value_counts().sort_index()
axes[0].bar(cluster_sizes.index, cluster_sizes.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Customer Cluster', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].set_title('Cluster Size Distribution', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Average Recency by Cluster
recency_by_cluster = final_customer_df.groupby("Customer_Cluster")["Recency"].mean()
axes[1].bar(recency_by_cluster.index, recency_by_cluster.values, color='coral', edgecolor='black')
axes[1].set_xlabel('Customer Cluster', fontsize=11)
axes[1].set_ylabel('Recency (days)', fontsize=11)
axes[1].set_title('Average Recency by Cluster', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Average Frequency by Cluster
freq_by_cluster = final_customer_df.groupby("Customer_Cluster")["Frequency"].mean()
axes[2].bar(freq_by_cluster.index, freq_by_cluster.values, color='lightgreen', edgecolor='black')
axes[2].set_xlabel('Customer Cluster', fontsize=11)
axes[2].set_ylabel('Frequency (baskets)', fontsize=11)
axes[2].set_title('Average Frequency by Cluster', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

# Plot 4: Average Basket Value by Cluster
basket_val_by_cluster = final_customer_df.groupby("Customer_Cluster")["Avg_Basket_Value"].mean()
axes[3].bar(basket_val_by_cluster.index, basket_val_by_cluster.values, color='gold', edgecolor='black')
axes[3].set_xlabel('Customer Cluster', fontsize=11)
axes[3].set_ylabel('Avg Basket Value (€)', fontsize=11)
axes[3].set_title('Average Basket Value by Cluster', fontsize=12, fontweight='bold')
axes[3].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("customer_cluster_profiles.png", dpi=150)
plt.show()

print("\n✓ Clustering Complete!")
print("✓ Visualizations saved:")
print("  - customer_k_selection_analysis.png")
print("  - customer_umap_clusters.png")
print("  - customer_cluster_profiles.png")

In [ ]:
# ...existing code...
from IPython.display import display, Markdown

# Ensure customer_summary & final_customer_df exist
try:
    summary = customer_summary.copy()
    dfc = final_customer_df.copy()
except NameError:
    print("Error: Εκτέλεσε πρώτα τα cells που φτιάχνουν 'customer_summary' και 'final_customer_df'."); raise

# Add counts if missing
counts = dfc["Customer_Cluster"].value_counts().sort_index()
if "Count" not in summary.columns:
    summary["Count"] = counts

# Show table for verification
print("\n--- Customer summary (per cluster) ---")
display(summary)

# Heuristic labeling (πρόταση — ελέγξτε)
labels = {}
avg_basket = summary["Avg_Basket_Value_Avg"].astype(float)
freq = summary["Frequency_Baskets_Avg"].astype(float)
rec = summary["Recency_Days_Avg"].astype(float)
cnt = summary["Count"].astype(int)

for i in summary.index:
    if cnt.loc[i] <= 3 and freq.loc[i] > summary["Frequency_Baskets_Avg"].max() * 0.8:
        labels[i] = "Outlier / B2B"
    elif avg_basket.loc[i] == avg_basket.max():
        labels[i] = "Stock‑Up (High‑value)"
    elif freq.loc[i] == freq.max():
        labels[i] = "Mainstream (High‑frequency)"
    elif rec.loc[i] == rec.max():
        labels[i] = "At‑Risk / Lapsed"
    else:
        labels[i] = "Premium / Other"

# Display proposed mapping
md = "### Προτεινόμενη αντιστοίχιση cluster → περιγραφή\n\n"
for k, v in labels.items():
    md += f"- Cluster {k}: **{v}**  \n"

display(Markdown(md))
# ...existing code...

In [ ]:
# ...existing code...
mapping = {
    0: "At-Risk / Lapsed",
    1: "Core Mainstream",
    2: "Stock-Up (High-value)",
    3: "High-frequency / Heavy shoppers"
}

# show actual counts + suggested labels
counts = final_customer_df["Customer_Cluster"].value_counts().sort_index()
print("Actual counts:", counts.to_dict())

# print suggested mapping with cluster metrics
for k, label in mapping.items():
    if k in customer_summary.index:
        row = customer_summary.loc[k]
        print(f"Cluster {k}: {label} | Count={int(row['Count'])} | Recency={row['Recency_Days_Avg']} | Freq={row['Frequency_Baskets_Avg']} | AvgBasket={row['Avg_Basket_Value_Avg']}")
    else:
        print(f"Cluster {k}: {label} | (no summary row)")

# Optionally update cluster_names used in plots
cluster_names = {k: v for k, v in mapping.items()}
print("\ncluster_names updated:", cluster_names)
# ...existing code...

Cluster 0 — At‑Risk / Lapsed

Κύρια χαρακτηριστικά: Count = 154, Recency ≈ 168.5 ημέρες, Μέση Συχνότητα ≈ 3.8 καλάθια, Μέση Αξία Καλαθιού ≈ 7.04€, Μέση μοναδικά είδη ανα καλάθι ≈ 12.9.
Ερμηνεία: μικρό δείγμα πελατών με υψηλή απομάκρυνση (πολύ μεγάλη Recency) και χαμηλή συχνότητα — πιθανοί εγκαταλελειμμένοι ή μη ενεργοί πελάτες.
Συστάσεις: στοχευμένες win‑back εκστρατείες (προσωποποιημένα κουπόνια, επανενεργοποίηση με προσφορές), απλές προτάσεις επιστροφής (έκπτωση στην επόμενη αγορά), παρακολούθηση response rate πριν ενσωμάτωση σε κύριες αναφορές.

Cluster 1 — Core Mainstream

Κύρια χαρακτηριστικά: Count = 1.373, Recency ≈ 131.9 ημέρες, Μέση Συχνότητα ≈ 10.8 καλάθια, Μέση Αξία Καλαθιού ≈ 9.36€, Μέση μοναδικά είδη ≈ 34.6.
Ερμηνεία: κύριος όγκος πελατών (core), σχετικά τακτικοί αγοραστές με μικρομεσαία αξία ανά καλάθι — η βασική εμπορική βάση.
Συστάσεις: βελτίωση ARPU με cross‑sell, bundles, προωθήσεις προϊόντων με μέση/υψηλή αξία, loyalty perks για διατήρηση και μικρές εξατομικευμένες προσφορές για αύξηση αξίας ανά επίσκεψη.

Cluster 2 — Stock‑Up (High‑value)

Κύρια χαρακτηριστικά: Count = 429, Recency ≈ 138.2 ημέρες, Μέση Συχνότητα ≈ 9.3 καλάθια, Μέση Αξία Καλαθιού ≈ 24.61€, Μέση μοναδικά είδη ≈ 63.1.
Ερμηνεία: πελάτες που κάνουν σχετικά συχνές αλλά μεγάλες αξιακά/ποσοτικά αγορές — σημαντικοί για συνολικά έσοδα.
Συστάσεις: διατήρηση με εξατομικευμένες προσφορές αξίας, μεγάλες προσφορές/κουπόνια για bulk αγορές, cross‑category bundles, ειδικές υπηρεσίες/προώθηση επαναλαμβανόμενων αγορών.

Cluster 3 — High‑frequency / Heavy shoppers (Potential VIP)

Κύρια χαρακτηριστικά: Count = 434, Recency ≈ 37.5 ημέρες, Μέση Συχνότητα ≈ 65.4 καλάθια, Μέση Αξία Καλαθιού ≈ 13.28€, Μέση μοναδικά είδη ≈ 201.4.
Ερμηνεία: πολύ ενεργοί, καθημερινοί/συχνoί αγοραστές με μεγάλο όγκο ειδών — πιθανώς καταστήματα/επαγγελματίες ή πολύ πιστοί heavy shoppers.
Συστάσεις: εξετάστε αν είναι B2B/internal· αν είναι καταναλωτές, προγράμματα VIP (προτεραιότητα, εξατομικευμένα οφέλη), ειδικές συμφωνίες, και παρακολούθηση για potential churn αν αυξήσει Recency.

Ερώτηση: Ποιες CustomCategories έχουν την υψηλότερη και χαμηλότερη πιθανότητα να εμφανιστούν μαζί με τις 5 κορυφαίες κατηγορίες του καταστήματος (με βάση τον όγκο συναλλαγών) και πώς διαφέρει η μέση αξία συναλλαγής όταν συμπεριλαμβάνονται αυτές οι δευτερεύουσες κατηγορίες;



In [ ]:
!pip install apyori

import pandas as pd
from apyori import apriori
import matplotlib.pyplot as plt
import numpy as np

# Υποθέτουμε ότι το basket_matrix (One-hot encoded Baskets x Categories) είναι διαθέσιμο
# και το df_pos (το καθαρό, merged POS DataFrame) είναι διαθέσιμο

# =========================================================
# 1. Βήμα: Εύρεση Top 5 Κατηγοριών με βάση την Συχνότητα Εμφάνισης
# =========================================================

# Χρησιμοποιούμε το αρχικό one-hot encoded matrix χωρίς το 'Total_categories'
basket_data = basket_matrix.drop(columns=['Total_categories', 'cluster'], errors='ignore')

# Συχνότητα εμφάνισης (Support)
category_support = basket_data.sum(axis=0).sort_values(ascending=False)
top_5_categories = category_support.head(5).index.tolist()

print("\n" + "="*80)
print(f"--- Top 5 CustomCategories (by Basket Count): {top_5_categories} ---")
print("="*80)

# =========================================================
# 2. Βήμα: Υπολογισμός Μέσου Όρου Αξίας Καλαθιού ανά Κατηγορία
# =========================================================

# Συνδέουμε την αξία του καλαθιού με το basket_matrix
# 1. Υπολογισμός συνολικής αξίας ανά Basket_ID
basket_values = df_pos.groupby('Basket_ID')['Value'].sum().rename('Basket_Value')

# 2. Ενοποίηση με το basket_data (μόνο τα κοινά Basket_IDs)
basket_analysis_df = basket_data.join(basket_values, how='inner')
basket_analysis_df = basket_analysis_df.dropna(subset=['Basket_Value'])

results = []

for main_cat in top_5_categories:
    # Κατηγορίες που εμφανίζονται μαζί με την main_cat
    co_occurrence_mask = (basket_analysis_df[main_cat] == 1)

    # 3. Βασική Μέση Αξία (Baseline)
    baseline_value = basket_analysis_df['Basket_Value'].mean()

    # 4. Μέση Αξία για καλάθια που περιέχουν την main_cat
    main_cat_value = basket_analysis_df.loc[co_occurrence_mask, 'Basket_Value'].mean()

    # 5. Υπολογισμός metrics για δευτερεύουσες κατηγορίες (Co-occurring Categories)
    for other_cat in basket_data.columns:
        if other_cat == main_cat:
            continue
        
        # 5.1 Μάσκα: Περιέχει και τα δύο (main_cat & other_cat)
        both_mask = co_occurrence_mask & (basket_analysis_df[other_cat] == 1)

        # 5.2 Ποσοστό εμφάνισης (Support Count)
        co_occurrence_count = both_mask.sum()
        
        # 5.3 Πιθανότητα Εμφάνισης (Confidence: P(Other | Main))
        # Πόσα από τα καλάθια με την main_cat περιέχουν και την other_cat
        if co_occurrence_mask.sum() > 0:
             confidence = co_occurrence_count / co_occurrence_mask.sum()
        else:
             confidence = 0

        # 5.4 Μέση Αξία Καλαθιού με Συν-εμφάνιση
        if co_occurrence_count > 0:
            avg_co_value = basket_analysis_df.loc[both_mask, 'Basket_Value'].mean()
        else:
            avg_co_value = np.nan
        
        # 5.5 Σύγκριση (Lift in Value)
        # Πόσο αυξάνεται η αξία από την συν-εμφάνιση σε σχέση με την main_cat_value
        lift_in_value = (avg_co_value / main_cat_value) - 1 if main_cat_value > 0 else np.nan


        results.append({
            'MainCategory': main_cat,
            'Co_Category': other_cat,
            'Confidence': confidence,
            'Co_Basket_Value_Avg': avg_co_value,
            'Value_Lift_%': lift_in_value * 100
        })

# 6. Τελικό DataFrame και Εμφάνιση
co_occurrence_df = pd.DataFrame(results).dropna(subset=['Co_Basket_Value_Avg'])
co_occurrence_df = co_occurrence_df[co_occurrence_df['Confidence'] > 0] # Αφαίρεση 0 confidence

# Εμφάνιση Top & Bottom 3 Συν-εμφανίσεις για κάθε Main Category
for main_cat in top_5_categories:
    print("\n" + "="*80)
    print(f"--- Ανάλυση Συν-εμφάνισης για την Κύρια Κατηγορία: {main_cat} ---")
    
    subset = co_occurrence_df[co_occurrence_df['MainCategory'] == main_cat]
    
    # Πιο πιθανές συν-εμφανίσεις (High Confidence)
    top_confidence = subset.sort_values(by='Confidence', ascending=False).head(3)
    print("\n**TOP 3 Co-Categories (Highest Confidence)**:")
    print(top_confidence[['Co_Category', 'Confidence', 'Co_Basket_Value_Avg']].round(2).to_markdown(index=False))

    # Συν-εμφανίσεις που αυξάνουν περισσότερο την αξία (Highest Value Lift)
    top_value_lift = subset.sort_values(by='Value_Lift_%', ascending=False).head(3)
    print("\n**TOP 3 Co-Categories (Highest Value Lift %)**:")
    print(top_value_lift[['Co_Category', 'Value_Lift_%', 'Co_Basket_Value_Avg']].round(2).to_markdown(index=False))

    print(f"\nΜέση Αξία Καλαθιού (μεμονωμένα): {main_cat}: {main_cat_value:.2f}€")

Η απάντηση έδωσε κανόνες συσχέτισης (Association Rules) για τα κορυφαία προϊόντα, οι οποίοι είναι κρίσιμοι για τη βελτιστοποίηση της διάταξης του καταστήματος, τις προωθήσεις (bundles) και τις συστάσεις προϊόντων.

Ερώτηση: Πώς κατανέμεται η αγοραστική δύναμη για τις κορυφαίες 4 κατηγορίες που αυξάνουν την αξία του καλαθιού (Value Lift), μεταξύ των Customer Clusters; Δηλαδή, ποιο Customer Cluster (π.χ. Stock-Up, Heavy Shopper) είναι ο πιο σημαντικός αγοραστής των Βιολογικών και των Κύβων/Χαλβάδων (οι κατηγορίες με τον υψηλότερο Value Lift);

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Υποθέτουμε ότι το df_pos (το καθαρό, merged POS DataFrame) είναι διαθέσιμο
# και το final_customer_df (με τη στήλη 'Customer_Cluster') είναι διαθέσιμο

# 1. Βήμα: Ορισμός των Στρατηγικών Κατηγοριών Υψηλής Αξίας
# Αυτές οι κατηγορίες βρέθηκαν ότι έχουν τον υψηλότερο Value Lift %
# Βιολογικα, Κυβοι, Χαλβαδες Ταχινι, Κατεψυγμενα Κρεας & Γευματα
strategic_categories = [
    'Βιολογικα',
    'Κυβοι',
    'Χαλβαδες Ταχινι',
    'Κατεψυγμενα Κρεας & Γευματα'
]

# 2. Βήμα: Ενοποίηση Customer Cluster με το DataFrame συναλλαγών (df_pos)
# Χρειαζόμαστε το df_pos merged με το LoyaltyCard_ID και μετά το Cluster
# (Χρησιμοποιούμε το df_loyal που δημιουργήθηκε σε προηγούμενα βήματα, το οποίο περιέχει
# τις στήλες 'LoyaltyCard_ID', 'CustomCategory', 'Value' και 'Customer_Cluster')

# Merge df_pos with Customer Clusters (επαναλαμβάνουμε για σιγουριά)
df_loyal = df_pos.dropna(subset=['LoyaltyCard_ID']).merge(
    final_customer_df[['Customer_Cluster']],
    on='LoyaltyCard_ID',
    how='inner'
).copy()

# Filter μόνο για τις στρατηγικές κατηγορίες
df_strategic = df_loyal[df_loyal['CustomCategory'].isin(strategic_categories)].copy()

# 3. Βήμα: Υπολογισμός συνολικής αξίας ανά Κατηγορία και Customer Cluster
value_per_category_cluster = df_strategic.groupby(
    ['Customer_Cluster', 'CustomCategory']
)['Value'].sum().reset_index()

# 4. Βήμα: Υπολογισμός του Global Value Share (Ποιος αγοράζει τι)
# Ποιο Customer Cluster έχει το μεγαλύτερο μερίδιο αξίας για κάθε στρατηγική κατηγορία

# Συνολική αξία ανά στρατηγική κατηγορία
total_value_per_cat = df_strategic.groupby('CustomCategory')['Value'].sum().reset_index().rename(
    columns={'Value': 'Total_Cat_Value'}
)

# Ενοποίηση και υπολογισμός Value Share (μερίδιο του Cluster επί του συνόλου της κατηγορίας)
category_cluster_share = value_per_category_cluster.merge(
    total_value_per_cat, on='CustomCategory', how='left'
)
category_cluster_share['Value_Share'] = (
    category_cluster_share['Value'] / category_cluster_share['Total_Cat_Value']
)

# 5. Βήμα: Δημιουργία Pivot Table για Οπτικοποίηση
pivot_table = category_cluster_share.pivot_table(
    index='CustomCategory',
    columns='Customer_Cluster',
    values='Value_Share',
    fill_value=0
)

# --- Εμφάνιση Πίνακα για Ακρίβεια ---
print("\n" + "="*80)
print(f"--- Customer Cluster Value Share for High-Lift Categories ---")
print("Ποσοστό της συνολικής αξίας της κατηγορίας που αγοράζει κάθε Cluster.")
print("="*80)
print(pivot_table.T.round(3).to_markdown(floatfmt=".1%"))


# 6. Βήμα: Οπτικοποίηση (Stacked Bar Plot)
plt.figure(figsize=(10, 7))

pivot_table.T.plot(
    kind='bar',
    stacked=True,
    colormap='Spectral', # Χρώματα που διαφοροποιούν τα clusters
    ax=plt.gca(),
    edgecolor='black'
)

# --- Βελτίωση Γραφήματος ---
plt.title(
    'Customer Cluster Value Share for High-Value-Lift Categories',
    fontsize=14,
    fontweight='bold'
)
plt.ylabel('Share of Total Category Value', fontsize=12)
plt.xlabel('Customer Cluster', fontsize=12)
plt.xticks(
    ticks=pivot_table.T.index,
    labels=[f"C{i}\n({final_customer_df['Customer_Cluster'].value_counts().get(i, 0)} Count)" for i in pivot_table.T.index],
    rotation=0
)
plt.yticks(np.arange(0, 1.1, 0.1), [f'{i*100:.0f}%' for i in np.arange(0, 1.1, 0.1)])

plt.legend(
    title='High-Lift Category',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show() # 

print("\n✓ Οπτικοποίηση της Στόχευσης Ολοκληρώθηκε.")